In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/syxlicheng/heartdatabase/EchoNet-Dynamic/FileList.csv")

print(df.head())

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

mask_path = "/kaggle/input/datasets/harshithabhaskaran/frames-masks/masks"

videos = {}

for file in os.listdir(mask_path):

    if not file.endswith(".png"):
        continue

    video_id = "_".join(file.split("_")[:-1])

    if video_id not in videos:
        videos[video_id] = []

    videos[video_id].append(file)

print("Total videos found:", len(videos))

results = []

for vid in tqdm(videos):

    files = videos[vid]

    if len(files) != 2:
        continue

    areas = []

    for f in files:

        img = cv2.imread(os.path.join(mask_path, f), 0)

        area = np.sum(img > 0)   # FIX HERE

        areas.append(area)

    ED_area = max(areas)
    ES_area = min(areas)

# convert area → volume approximation
    EDV = ED_area ** 1.5
    ESV = ES_area ** 1.5

    EF = (EDV - ESV) / EDV * 100

    results.append({
        "Video_ID": vid,
        "EF_pred": EF
    })

pred_df = pd.DataFrame(results)

print("Predictions:", len(pred_df))
pred_df.head()

In [ ]:
output_path = "/kaggle/working/ef_predictions.csv"

pred_df.to_csv(output_path, index=False)

print("EF predictions saved to:", output_path)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error
import pandas as pd
import matplotlib.pyplot as plt

# Load ground truth
gt = pd.read_csv("/kaggle/input/datasets/syxlicheng/heartdatabase/EchoNet-Dynamic/FileList.csv")

# Merge predictions with GT
merged = pred_df.merge(gt, left_on="Video_ID", right_on="FileName")

print("Matched samples:", len(merged))

# Metrics
mae = mean_absolute_error(merged["EF"], merged["EF_pred"])
r2 = r2_score(merged["EF"], merged["EF_pred"])

print("MAE:", round(mae,2))
print("R2 Score:", round(r2,3))

# Scatter plot
plt.figure(figsize=(6,6))
plt.scatter(merged["EF"], merged["EF_pred"], alpha=0.4)

plt.plot([0,100],[0,100],'r--')

plt.xlabel("Ground Truth EF")
plt.ylabel("Predicted EF")
plt.title(f"EF Prediction (R2={r2:.3f})")

plt.grid(True)
plt.show()

In [ ]:
import pandas as pd

pred = pd.read_csv("/kaggle/working/ef_predictions.csv")
gt = pd.read_csv("/kaggle/input/datasets/syxlicheng/heartdatabase/EchoNet-Dynamic/FileList.csv")

# rename if needed
gt = gt.rename(columns={"FileName":"Video_ID", "EF":"EF_true"})

df = pred.merge(gt, on="Video_ID")

print("Matched samples:", len(df))

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(df["EF_true"], df["EF_pred"])
r2 = r2_score(df["EF_true"], df["EF_pred"])

print("MAE:", mae)
print("R2:", r2)

In [ ]:
import numpy as np

errors = abs(df["EF_true"] - df["EF_pred"])

boot = []
for i in range(1000):
    sample = np.random.choice(errors, size=len(errors), replace=True)
    boot.append(sample.mean())

ci_low = np.percentile(boot, 2.5)
ci_high = np.percentile(boot, 97.5)

print("95% CI for MAE:", ci_low, "-", ci_high)

In [ ]:
import matplotlib.pyplot as plt

mean = (df["EF_true"] + df["EF_pred"]) / 2
diff = df["EF_pred"] - df["EF_true"]

plt.scatter(mean, diff)
plt.axhline(diff.mean(), linestyle="--")
plt.axhline(diff.mean() + 1.96*diff.std(), linestyle="--")
plt.axhline(diff.mean() - 1.96*diff.std(), linestyle="--")

plt.xlabel("Mean EF")
plt.ylabel("Difference (Pred - True)")
plt.title("Bland-Altman Plot")
plt.show()

In [ ]:
def ef_group(ef):
    if ef < 40:
        return "HFrEF"
    elif ef < 50:
        return "HFmrEF"
    else:
        return "HFpEF"

df["group"] = df["EF_true"].apply(ef_group)

for g in df["group"].unique():
    sub = df[df["group"] == g]
    mae = abs(sub["EF_true"] - sub["EF_pred"]).mean()
    print(g, "MAE:", mae)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df["EF_true"], df["EF_pred"], alpha=0.5)

plt.plot([10,80],[10,80],'r--')   # perfect prediction line

plt.xlabel("True EF")
plt.ylabel("Predicted EF")
plt.title("Predicted vs True EF")

plt.show()

In [ ]:
error = df["EF_pred"] - df["EF_true"]

plt.hist(error, bins=50)

plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.title("Distribution of EF Prediction Errors")

plt.show()